# Automatic summarization of medical records with Hugging Face

Pipeline: JSON of simulated records → text per patient → summary with a pre-trained encoder-decoder model → output JSON.

Only simulated data is used. The models are already trained: here we only do *inference* (generation) on the GPU.

## 1. Installing the libraries
`transformers` contains models and tokenizers, `torch` does the computation on the GPU, `sentencepiece` is needed by the T5 tokenizer.

In [1]:
%pip install --quiet transformers==4.57.6 sentencepiece torch



[notice] A new release of pip is available: 26.2.1 -> 26.3
[notice] To update, run: /home/ubugab/ml-projects/tf-env/bin/python3 -m pip install --upgrade pip
Note: you may need to restart the kernel to use updated packages.


## 2. GPU check
We check that PyTorch can see the NVIDIA GPU (CUDA). If there is none, we stop with an error: generation on the CPU would be much slower.

In [2]:
import os
os.environ['TORCH_DISABLE_NATIVE_JIT'] = '1'

import torch

if not torch.cuda.is_available():
    raise RuntimeError('CUDA not available: an NVIDIA GPU is required.')

device = torch.device('cuda')
print('GPU in use:', torch.cuda.get_device_name(0))


GPU in use: NVIDIA GeForce RTX 4060


## 3. Loading the dataset
We download the JSON file with the simulated records and turn it into a Python list: each element is a patient.

In [3]:
import json
from urllib.request import urlopen

URL = (
    'https://raw.githubusercontent.com/Profession-AI/progetti-deeplearning/'
    'refs/heads/main/Sintesi%20automatica%20di%20cartelle%20cliniche%20di%20'
    'un%E2%80%99azienda%20ospedaliera/hospital_records.json'
)

with urlopen(URL, timeout=30) as response:
    patients = json.loads(response.read().decode('utf-8'))

print('Patients loaded:', len(patients))


Patients loaded: 10


## 4. Data exploration
We look at how many stays each patient has, what a record looks like and whether any values are missing.

In [4]:
FIELDS = ['admission_date', 'discharge_date', 'diagnosis', 'anamnesis',
          'prognosis', 'medications', 'test_results']

missing = 0
for patient in patients:
    stays = patient['hospitalizations']
    print(patient['patient_name'], '->', len(stays), 'stays')
    for stay in stays:
        for field in FIELDS:
            if not stay.get(field):
                missing += 1
                print('  Missing value:', field)

print('\nTotal missing values:', missing)
print('\nExample record:')
print(json.dumps(patients[0], indent=2))


Patient 1 -> 2 stays
Patient 2 -> 3 stays
Patient 3 -> 1 stays
Patient 4 -> 2 stays
Patient 5 -> 3 stays
Patient 6 -> 3 stays
Patient 7 -> 2 stays
Patient 8 -> 1 stays
Patient 9 -> 2 stays
Patient 10 -> 2 stays

Total missing values: 0

Example record:
{
  "patient_name": "Patient 1",
  "hospitalizations": [
    {
      "admission_date": "2023-04-11",
      "discharge_date": "2023-04-14",
      "diagnosis": "Pneumonia",
      "anamnesis": "No relevant anamnesis",
      "prognosis": "Requires further monitoring",
      "medications": [
        "Captopril",
        "Prednisone",
        "Ciprofloxacin",
        "Ibuprofen"
      ],
      "test_results": [
        "CT scan shows inflammation"
      ]
    },
    {
      "admission_date": "2023-10-16",
      "discharge_date": "2023-10-27",
      "diagnosis": "Bronchitis",
      "anamnesis": "No prior conditions",
      "prognosis": "Surgery recommended",
      "medications": [
        "Ciprofloxacin",
        "Captopril"
      ],
      "tes

## 5. Building the text of each patient
For each patient we sort the stays by admission date and write one block of text per stay, with clear labels (`Diagnosis:`, `Prognosis:`, ...). We keep the blocks in a list: we will need them to split texts that are too long without cutting a stay in half. An empty field becomes `not recorded`.

In [5]:
def build_blocks(patient):
    stays = sorted(patient['hospitalizations'], key=lambda stay: stay['admission_date'])
    blocks = []
    for number, stay in enumerate(stays, start=1):
        medications = '; '.join(stay.get('medications') or []) or 'not recorded'
        tests = '; '.join(stay.get('test_results') or []) or 'not recorded'
        lines = [
            f'Hospitalization {number}.',
            f'Admission date: {stay.get("admission_date") or "not recorded"}.',
            f'Discharge date: {stay.get("discharge_date") or "not recorded"}.',
            f'Diagnosis: {stay.get("diagnosis") or "not recorded"}.',
            f'Medical history: {stay.get("anamnesis") or "not recorded"}.',
            f'Prognosis: {stay.get("prognosis") or "not recorded"}.',
            f'Medications: {medications}.',
            f'Test results: {tests}.',
        ]
        blocks.append('\n'.join(lines))
    return blocks


texts = {patient['patient_name']: build_blocks(patient) for patient in patients}

print('\n'.join(texts['Patient 2']))


Hospitalization 1.
Admission date: 2023-01-18.
Discharge date: 2023-01-27.
Diagnosis: Arthritis.
Medical history: History of hypertension.
Prognosis: Requires further monitoring.
Medications: Prednisone; Amoxicillin; Paracetamol.
Test results: Fracture confirmed by X-ray; CT scan shows inflammation.
Hospitalization 2.
Admission date: 2023-04-07.
Discharge date: 2023-04-20.
Diagnosis: Fracture of leg.
Medical history: History of hypertension.
Prognosis: Positive recovery expected.
Medications: Paracetamol; Aspirin; Ciprofloxacin.
Test results: CT scan shows inflammation; MRI indicates tissue damage.
Hospitalization 3.
Admission date: 2023-07-06.
Discharge date: 2023-07-07.
Diagnosis: Diabetes management.
Medical history: Diabetic patient.
Prognosis: Complete recovery.
Medications: Ciprofloxacin; Prednisone; Paracetamol.
Test results: MRI indicates tissue damage.


## 6. Tokenizers and length in tokens
A model reads *tokens*, not words. We use 1024 tokens as the input limit for BART and 512 as the operational limit for T5 (which does not have the same absolute-position limit as BART). T5 also wants the prefix `summarize: ` in front of the text. We count the tokens, special ones included, with the tokenizer of each model.

In [6]:
from transformers import AutoTokenizer

MODELS = {
    'facebook/bart-large-cnn': {'prefix': '', 'max_tokens': 1024},
    't5-small': {'prefix': 'summarize: ', 'max_tokens': 512},
}

tokenizers = {name: AutoTokenizer.from_pretrained(name) for name in MODELS}


def count_tokens(text, model_name):
    prefix = MODELS[model_name]['prefix']
    return len(tokenizers[model_name](prefix + text, truncation=False, verbose=False)['input_ids'])


for model_name, info in MODELS.items():
    print(model_name, '- limit', info['max_tokens'], 'tokens')
    for patient_name, blocks in texts.items():
        n_tokens = count_tokens('\n'.join(blocks), model_name)
        print(f'  {patient_name}: {n_tokens} tokens')


/home/ubugab/ml-projects/tf-env/lib/python3.12/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm
/home/ubugab/ml-projects/tf-env/lib/python3.12/site-packages/sklearn/utils/_param_validation.py:14: UserWarning: A NumPy version >=1.23.5 and <2.3.0 is required for this version of SciPy (detected version 2.5.4)
  from scipy.sparse import csr_array, issparse


facebook/bart-large-cnn - limit 1024 tokens
  Patient 1: 174 tokens
  Patient 2: 266 tokens
  Patient 3: 87 tokens
  Patient 4: 159 tokens
  Patient 5: 268 tokens
  Patient 6: 252 tokens
  Patient 7: 164 tokens
  Patient 8: 84 tokens
  Patient 9: 183 tokens
  Patient 10: 174 tokens
t5-small - limit 512 tokens
  Patient 1: 162 tokens
  Patient 2: 251 tokens
  Patient 3: 80 tokens
  Patient 4: 145 tokens
  Patient 5: 252 tokens
  Patient 6: 233 tokens
  Patient 7: 155 tokens
  Patient 8: 82 tokens
  Patient 9: 170 tokens
  Patient 10: 167 tokens


## 7. Splitting texts that are too long
If the text exceeds the limit, we split it into parts. We add one stay at a time; when the limit would be exceeded, we start a new part. Only if a single stay is too long do we split it between words, without truncating information. Since all our patients are within the limit, we also test artificially long texts. The partial summaries are joined into one paragraph: it may be less fluent and lose connections between episodes.

In [7]:
def split_into_parts(blocks, model_name):
    max_tokens = MODELS[model_name]['max_tokens']
    if not blocks or any(not block.strip() for block in blocks):
        raise ValueError('Stays with non-empty text are required.')
    short_blocks = []
    for block in blocks:
        if count_tokens(block, model_name) <= max_tokens:
            short_blocks.append(block)
            continue
        words = []
        for word in block.split():
            if count_tokens(word, model_name) > max_tokens:
                raise ValueError('A single word exceeds the model limit.')
            if words and count_tokens(' '.join(words + [word]), model_name) > max_tokens:
                short_blocks.append(' '.join(words))
                words = []
            words.append(word)
        short_blocks.append(' '.join(words))

    parts = []
    current = []
    for block in short_blocks:
        candidate = current + [block]
        if current and count_tokens('\n'.join(candidate), model_name) > max_tokens:
            parts.append('\n'.join(current))
            current = [block]
        else:
            current = candidate
    parts.append('\n'.join(current))
    return parts


long_blocks = texts['Patient 2'] * 8
for model_name in MODELS:
    parts = split_into_parts(long_blocks, model_name)
    sizes = [count_tokens(part, model_name) for part in parts]
    assert all(size <= MODELS[model_name]['max_tokens'] for size in sizes)
    assert '\n'.join(parts) == '\n'.join(long_blocks)
    print(model_name, '->', len(parts), 'parts, tokens per part:', sizes)
    long_stay = ' '.join(long_blocks)
    parts = split_into_parts([long_stay], model_name)
    assert all(count_tokens(part, model_name) <= MODELS[model_name]['max_tokens'] for part in parts)
    assert ' '.join(' '.join(parts).split()) == ' '.join(long_stay.split())
    print('The single long stay is also split without losing words.')


facebook/bart-large-cnn -> 3 parts, tokens per part: [977, 969, 177]
The single long stay is also split without losing words.
t5-small -> 4 parts, tokens per part: [499, 499, 499, 499]
The single long stay is also split without losing words.


## 8. Functions to generate the summary
- `load_model` loads the model and moves it to the GPU.
- `summarize` turns the text into tokens, generates the summary and converts it back to text.
- `summarize_patient` splits the text if needed, summarizes each part and joins them.

Starting generation parameters: `min_length` and `max_length` (summary length in tokens), `num_beams` (how many candidate sequences the model evaluates), `no_repeat_ngram_size` (avoids repeating groups of 3 tokens), `length_penalty` and `early_stopping`. We set them explicitly because the models have different default values.

In [8]:
from transformers import AutoModelForSeq2SeqLM

PARAMS = {
    'min_length': 30,
    'max_length': 160,
    'num_beams': 4,
    'no_repeat_ngram_size': 3,
    'length_penalty': 1.0,
    'early_stopping': True,
    'do_sample': False,
}


def load_model(model_name):
    return AutoModelForSeq2SeqLM.from_pretrained(model_name).to(device).eval()


def summarize(text, model_name, model, **params):
    tokenizer = tokenizers[model_name]
    inputs = tokenizer(MODELS[model_name]['prefix'] + text, return_tensors='pt').to(device)
    if inputs['input_ids'].shape[1] > MODELS[model_name]['max_tokens']:
        raise ValueError('Text too long: it must be split before summarizing.')
    with torch.inference_mode():
        output = model.generate(**inputs, **params)
    return ' '.join(tokenizer.decode(output[0], skip_special_tokens=True).split())


def summarize_patient(blocks, model_name, model, **params):
    parts = split_into_parts(blocks, model_name)
    return ' '.join(summarize(part, model_name, model, **params) for part in parts)


## 9. Comparing BART and T5
We load one model at a time, summarize three patients and measure the generation time. We run one unmeasured warm-up generation, because the first call on the GPU is always slower. At the end we free the GPU memory.

In [9]:
import time

COMPARISON_PATIENTS = ['Patient 3', 'Patient 2', 'Patient 9']
comparison = []

for model_name in MODELS:
    print('Loading', model_name)
    model = load_model(model_name)
    summarize_patient(texts['Patient 3'], model_name, model, **PARAMS)

    for patient_name in COMPARISON_PATIENTS:
        torch.cuda.synchronize()
        start = time.perf_counter()
        summary = summarize_patient(texts[patient_name], model_name, model, **PARAMS)
        torch.cuda.synchronize()
        seconds = time.perf_counter() - start
        comparison.append({
            'patient': patient_name,
            'model': model_name,
            'summary': summary,
            'seconds': seconds,
        })

    del model
    torch.cuda.empty_cache()

print('Comparison completed.')


Loading facebook/bart-large-cnn


I0000 00:00:1794824032.796551     866 cpu_feature_guard.cc:227] This TensorFlow binary is optimized to use available CPU instructions in performance-critical operations.
To enable the following instructions: AVX2 FMA, in other operations, rebuild TensorFlow with the appropriate compiler flags.


Loading t5-small
Comparison completed.


## 10. Reading the comparison
For each patient we show the original record and the summary of each model with the time taken. We compare diagnoses, medications, test results and prognosis: speed alone is not enough to choose.

In [10]:
for patient_name in COMPARISON_PATIENTS:
    print('=' * 80)
    print(patient_name, '- ORIGINAL RECORD')
    print('\n'.join(texts[patient_name]))
    for item in comparison:
        if item['patient'] == patient_name:
            print(f'\n{item["model"]} ({item["seconds"]:.2f} s):')
            print(item['summary'])

for model_name in MODELS:
    times = [item['seconds'] for item in comparison if item['model'] == model_name]
    print(f'\nAverage time {model_name}: {sum(times) / len(times):.2f} s')


Patient 3 - ORIGINAL RECORD
Hospitalization 1.
Admission date: 2023-06-26.
Discharge date: 2023-07-04.
Diagnosis: Hypertension.
Medical history: Frequent respiratory issues.
Prognosis: Complete recovery.
Medications: Amoxicillin; Metformin; Paracetamol; Captopril.
Test results: CT scan shows inflammation.

facebook/bart-large-cnn (0.75 s):
Prognosis: Complete recovery. Amoxicillin; Metformin; Paracetamol; Captopril. CT scan shows inflammation.

t5-small (0.69 s):
admission date: 2023-06-26; Discharge date: 2023-07-04. Medications: Amoxicillin; Metformin; Paracetamol; Captopril.
Patient 2 - ORIGINAL RECORD
Hospitalization 1.
Admission date: 2023-01-18.
Discharge date: 2023-01-27.
Diagnosis: Arthritis.
Medical history: History of hypertension.
Prognosis: Requires further monitoring.
Medications: Prednisone; Amoxicillin; Paracetamol.
Test results: Fracture confirmed by X-ray; CT scan shows inflammation.
Hospitalization 2.
Admission date: 2023-04-07.
Discharge date: 2023-04-20.
Diagnosis: 

## 11. Hyperparameter experiment
With BART on `Patient 2` we compare three configurations:
- **base**: the starting one;
- **short**: shorter summary and fewer candidates (`num_beams=2`);
- **sampling**: the model picks randomly among the most likely tokens (`do_sample`, `temperature`, `top_p`). The result is more varied, but also more at risk of errors. We fix the seed so the experiment can be repeated.

In [11]:
EXPERIMENT_MODEL = 'facebook/bart-large-cnn'

configs = {
    'base': PARAMS,
    'short': dict(PARAMS, min_length=10, max_length=60, num_beams=2),
    'sampling': {
        'min_length': 30,
        'max_length': 160,
        'num_beams': 1,
        'do_sample': True,
        'temperature': 0.8,
        'top_p': 0.9,
        'no_repeat_ngram_size': 3,
    },
}

model = load_model(EXPERIMENT_MODEL)
print('\n'.join(texts['Patient 2']))

for config_name, params in configs.items():
    torch.manual_seed(42)
    summary = summarize_patient(texts['Patient 2'], EXPERIMENT_MODEL, model, **params)
    print(f'\n--- {config_name} ---')
    print(summary)

del model
torch.cuda.empty_cache()


Hospitalization 1.
Admission date: 2023-01-18.
Discharge date: 2023-01-27.
Diagnosis: Arthritis.
Medical history: History of hypertension.
Prognosis: Requires further monitoring.
Medications: Prednisone; Amoxicillin; Paracetamol.
Test results: Fracture confirmed by X-ray; CT scan shows inflammation.
Hospitalization 2.
Admission date: 2023-04-07.
Discharge date: 2023-04-20.
Diagnosis: Fracture of leg.
Medical history: History of hypertension.
Prognosis: Positive recovery expected.
Medications: Paracetamol; Aspirin; Ciprofloxacin.
Test results: CT scan shows inflammation; MRI indicates tissue damage.
Hospitalization 3.
Admission date: 2023-07-06.
Discharge date: 2023-07-07.
Diagnosis: Diabetes management.
Medical history: Diabetic patient.
Prognosis: Complete recovery.
Medications: Ciprofloxacin; Prednisone; Paracetamol.
Test results: MRI indicates tissue damage.

--- base ---
Hospitalization 1. Admission date: 2023-01-18. Discharge date:2023-04-20. Diagnosis: Arthritis. Medications: Pre

The following generation flags are not valid and may be ignored: ['early_stopping', 'length_penalty']. Set `TRANSFORMERS_VERBOSITY=info` for more details.



--- short ---
Hospitalization 1: Arthritis. Hospitalization 2: Diabetes management. Hospitalized 3: Fracture of leg.

--- sampling ---
Hospitalization 1. Admitted to hospital with Arthritis, hypertension, and diabetes. Withdrawn from hospital. Spent time in hospital after discharge.


## 12. Model choice and summaries of all patients
### Comparison results and final choice

In the CUDA run of 16 November 2026, BART took on average **0.85 s** and T5 **0.75 s** on the three patients. These are few measurements: they are not a benchmark and may change on every run.

For Patient 3, BART keeps medications, test result and prognosis, while T5 mostly keeps dates and medications. Both omit the hypertension diagnosis. For Patient 2 both neglect most of the later stays; BART mixes up the dates, T5 produces a malformed date. So there is no clinically reliable winner.

We keep **BART as the model of the educational prototype**: it is specialized in summarization and, in the Patient 3 example, it keeps more categories of information. This local advantage does not prove general superiority; T5 remains the lighter and faster alternative in this test. Neither is approved here for clinical use.

### Parameter experiment results

- **Base:** keeps some details of the first stay but associates a discharge date of the second stay with the first.
- **Short:** lists three diagnoses, but swaps the second and third episodes and omits therapies and test results. Shorter does not mean more faithful.
- **Sampling:** produces phrases such as `Withdrawn from hospital` and `Spent time in hospital after discharge`, not supported by the record. Hypertension and diabetes, instead, are already present in the original: the problem is how they are joined and associated with the episodes.

We keep the **base parameters without sampling** as an educational compromise, not because they eliminate errors. The experiment changes several parameters together: it shows practical differences, but does not allow attributing them to a single parameter. Then we generate the summaries of the ten patients.

In [12]:
SELECTED_MODEL = 'facebook/bart-large-cnn'
SELECTED_PARAMS = PARAMS

model = load_model(SELECTED_MODEL)

results = []
for patient_name, blocks in texts.items():
    summary = summarize_patient(blocks, SELECTED_MODEL, model, **SELECTED_PARAMS)
    results.append({'patient_name': patient_name, 'summary': summary})
    print(patient_name, 'done')

del model
torch.cuda.empty_cache()


Patient 1 done
Patient 2 done
Patient 3 done
Patient 4 done
Patient 5 done
Patient 6 done
Patient 7 done
Patient 8 done
Patient 9 done
Patient 10 done


## 13. Saving the output JSON
We save the summaries in a JSON file. `ensure_ascii=False` keeps special characters readable.

In [13]:
OUTPUT_PATH = 'medical_record_summaries.json'

with open(OUTPUT_PATH, 'w', encoding='utf-8') as file:
    json.dump(results, file, ensure_ascii=False, indent=2)

print('Saved', len(results), 'summaries to', OUTPUT_PATH)


Saved 10 summaries to medical_record_summaries.json


## 14. Faithfulness check
We place the original record and the summary side by side. We check by hand that diagnoses, medications and test results are correct, that nothing is invented or omitted and that each item is in the right stay. Problem cases should be noted in the following cells.

In [14]:
summaries = {item['patient_name']: item['summary'] for item in results}

for patient_name in ['Patient 3', 'Patient 2', 'Patient 9']:
    print('=' * 80)
    print(patient_name, '- ORIGINAL RECORD')
    print('\n'.join(texts[patient_name]))
    print('\nGENERATED SUMMARY')
    print(summaries[patient_name])


Patient 3 - ORIGINAL RECORD
Hospitalization 1.
Admission date: 2023-06-26.
Discharge date: 2023-07-04.
Diagnosis: Hypertension.
Medical history: Frequent respiratory issues.
Prognosis: Complete recovery.
Medications: Amoxicillin; Metformin; Paracetamol; Captopril.
Test results: CT scan shows inflammation.

GENERATED SUMMARY
Prognosis: Complete recovery. Amoxicillin; Metformin; Paracetamol; Captopril. CT scan shows inflammation.
Patient 2 - ORIGINAL RECORD
Hospitalization 1.
Admission date: 2023-01-18.
Discharge date: 2023-01-27.
Diagnosis: Arthritis.
Medical history: History of hypertension.
Prognosis: Requires further monitoring.
Medications: Prednisone; Amoxicillin; Paracetamol.
Test results: Fracture confirmed by X-ray; CT scan shows inflammation.
Hospitalization 2.
Admission date: 2023-04-07.
Discharge date: 2023-04-20.
Diagnosis: Fracture of leg.
Medical history: History of hypertension.
Prognosis: Positive recovery expected.
Medications: Paracetamol; Aspirin; Ciprofloxacin.
Test 

### Problem cases observed

Manual check of a few examples from the saved run: it is not a medical validation or a full evaluation of the ten summaries.

| Case | Original | Observed result | Problem |
|---|---|---|---|
| Patient 3, BART | Hypertension; Frequent respiratory issues | Keeps medications, CT and prognosis, but not diagnosis and anamnesis | Omissions |
| Patient 2, BART | First discharge: 2023-01-27; second: 2023-04-20 | Associates the discharge 2023-04-20 with the first stay | Mixes up episodes |
| Patient 2, T5 | First discharge: 2023-01-27 | Writes `20,23-01-27` | Malformed date |
| Patient 9, BART | Discharges: 2023-03-16 and 2023-03-12 | Writes `2022-01-01` | Invented date |
| Patient 9, both | Heart failure and Arthritis in two stays | Do not report the arthritis diagnosis | Incomplete history |
| Patient 5 and Patient 6, BART | Three stays | Writes a fourth stay (`Hospitalised 4: Diabetes`) | Invented episode |
| Patient 8, BART | Appendicitis diagnosis | The diagnosis does not appear in the summary | Omission |

For Patient 9 the stays already overlap in the dataset: we keep the original dates without inventing corrections. Finding the right words is not enough: negations and associations between diagnosis, medications and episode must be checked. Having no reference summaries, we compute neither ROUGE nor a clinical accuracy percentage.

## 15. Limitations and risks in the clinical setting

**Hallucinations and omissions.** The model can invent dates, mix up stays or omit important diagnoses. We observed this even without sampling: deterministic generation does not mean truth. Imposing a minimum length can favor superfluous text.

**Language and domain.** Dataset and summaries remain in English. BART-CNN is specialized in news summarization, T5-small is a general multitask model. Neither is validated here for medical records in Italian. Translating would introduce another source of error.

**Long texts.** Avoiding truncation does not guarantee that the facts are preserved in the summary. Joining partial summaries can create repetitions or lose connections; splitting a stay can separate therapy and diagnosis.

**Privacy.** We use only public simulated data. Inference happens in the chosen environment, without an external generation API; Colab is still a cloud service. Do not upload real health data without proper authorizations and safeguards. Cell outputs and JSON files can also expose sensitive data. Real use would require data minimization, access control, file protection and an assessment of the applicable regulations, including the GDPR.

**Conclusion.** The prototype demonstrates the pipeline; it is not ready for clinical decisions. A professional's review and access to the original records are needed: the summary does not replace them.

## 16. Execution and delivery

The cells after the installation were run on GPU in WSL and the outputs are saved here. `medical_record_summaries.json`, in the project folder, contains ten elements with `patient_name` and `summary`. They are not summaries validated for clinical use.

**Google Colab is still to be verified:** upload this notebook, select a GPU runtime, run from the start and download the notebook and the JSON. The first cell installs the dependencies in the Colab environment. Times and results may change with hardware and versions: after a new run, re-check these observations.

No training or fine-tuning was done: we use pre-trained models in inference.